In [1]:
from functools import cache
from pathlib import Path

import pandas as pd
import google.auth
import geopandas as gpd

from calitp_data_analysis.gcs_pandas import GCSPandas
from calitp_data_analysis.sql import get_engine

# from shared_utils import bq_utils
# Initialize credentials and DB engine
credentials, project = google.auth.default()
db_engine = get_engine()

import _prep_crosswalk_ntd
import _load_gtfs_data
from update_vars import DIGEST_DICT

/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.11.10) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)
/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-packages/google/api_core/_python_version_support.py:273: FutureWarning: You are using a Python version (3.11.10) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2027-10-24). Please upgrade to the latest Python version, or at least Python 3.12, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)
/home/jovyan/gtfs-curator/.venv/lib/python3.11/site-pack

In [2]:
from update_vars import DIGEST_DICT as GTFS_DATA_DICT, abbrev_month, analysis_month 

In [3]:
@cache
def gcs_pandas():
    return GCSPandas()

In [4]:
pd.options.display.max_columns = 100
pd.options.display.float_format = "{:.2f}".format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)

## Operator = Why is October 2026 stuff missing? 

In [5]:
GCS_PATH = f"{GTFS_DATA_DICT.dir}processed/"

In [6]:
analysis_name = "Alameda-Contra Costa Transit District"

In [7]:
# Dataset 1
fct_monthly_routes_url = f"{GCS_PATH}{GTFS_DATA_DICT.route_map}.parquet"
fct_monthly_route_df = gpd.read_parquet(fct_monthly_routes_url,
                                        filters=[[("Analysis Name", "==", analysis_name)]],
                                        storage_options={"token": credentials.token}).reset_index()
fct_monthly_route_df['Number'] = fct_monthly_route_df.index

In [8]:
fct_monthly_route_df["Month First Day"].unique()

array([datetime.date(2026, 10, 1)], dtype=object)

In [9]:
# Dataset 2
schedule_rt_route_direction_summary_url = f"{GCS_PATH}{GTFS_DATA_DICT.schedule_rt_route_direction}.parquet"
schedule_rt_route_direction_summary_df = gcs_pandas().read_parquet(schedule_rt_route_direction_summary_url,
                         filters=[[("Analysis Name", "==", analysis_name)]])
one_month_schedule_rt_route_direction_summary_df = (schedule_rt_route_direction_summary_df.loc[schedule_rt_route_direction_summary_df["Day Type"] == "Weekday"]
    .sort_values(by = ["Date", "Route"], ascending = [False, True])
    .drop_duplicates(subset = ["Route"])
                                                   )

In [10]:
one_month_schedule_rt_route_direction_summary_df["Date"].unique()

<DatetimeArray>
['2026-11-01 00:00:00']
Length: 1, dtype: datetime64[ns]

In [11]:
# Dataset 3 
operator_hourly_summary_url = f"{GCS_PATH}{GTFS_DATA_DICT.hourly_day_type_summary}.parquet"
operator_hourly_summary_df = gcs_pandas().read_parquet(operator_hourly_summary_url,
                                             filters=[[("Analysis Name", "==", analysis_name),
                                                      ("Departure Hour", "<=", 24)]])

In [12]:
operator_hourly_summary_df["Date"].unique()

array(['10-2025', '11-2025', '12-2025', '01-2026', '02-2026', '03-2026',
       '04-2026', '05-2026', '06-2026', '07-2026', '08-2026', '09-2026',
       '10-2026'], dtype=object)

In [13]:
# Dataset 5 
operator_summary_url = f"{GCS_PATH}{GTFS_DATA_DICT.operator_summary}.parquet"
operator_df = (gcs_pandas().read_parquet(
    operator_summary_url,
    filters=[
        ("Day Type", "==", "Weekday"),
        ("Analysis Name", "==", analysis_name),
        ("VP Name", "!=", "None")],
)).sort_values(by = ["Date","Daily TU Trips"], ascending = [False,False]).drop_duplicates(subset = [
    "Analysis Name","Date"
])
operator_df["Pct TU Trips"] = operator_df["Pct TU Trips"].round(2)

In [14]:
operator_df["Date"].unique()

<DatetimeArray>
['2026-11-01 00:00:00', '2026-10-01 00:00:00', '2026-09-01 00:00:00']
Length: 3, dtype: datetime64[ns]

## Caltrans District, GTFS by Operator is wrong for D4

In [15]:
from update_vars import DIGEST_DICT, PROCESSED_GCS
import gcsfs
import _ct_report_utils as utils

In [16]:
district = "04 - Bay Area / Oakland"

In [17]:
f"{PROCESSED_GCS}{DIGEST_DICT.operator_summary}.parquet"

'gs://calitp-analytics-data/data-analyses/gtfs_digest/processed/fct_monthly_operator_summary.parquet'

In [18]:
f"{GCS_PATH}{GTFS_DATA_DICT.operator_summary}.parquet"

'gs://calitp-analytics-data/data-analyses/gtfs_digest/processed/fct_monthly_operator_summary.parquet'

In [19]:
# Dataset 4
operator_summary_url = f"{PROCESSED_GCS}{DIGEST_DICT.operator_summary}.parquet"

operator_df = pd.read_parquet(
    operator_summary_url,
    filesystem = gcsfs.GCSFileSystem(),
    filters=[
        ("Caltrans District", "==", district),
        ("Day Type", "==", "Weekday")
    ],
).sort_values(by = ["Analysis Name", "Date"], ascending = [False,False]).drop_duplicates(subset = ["Analysis Name"])

In [20]:
operator_df

,Date,Analysis Name,Caltrans District,VP Name,TU Name,N Trips,Day Type,Daily Trips,Ttl Service Hours,N Routes,N Days,N Shapes,N Stops,Daily Arrivals,VP Messages Per Minute,N VP Trips,Daily VP Trips,Pct VP Trips,TU Messages Per Minute,N TU Trips,Daily TU Trips,Pct TU Trips
60,2026-11-01,Western Contra Costa Transit Authority,04 - Bay Area / Oakland,Bay Area 511 WestCAT VehiclePositions,Bay Area 511 WestCAT TripUpdates,4568,Weekday,326.30,2520.50,11,14,40,218,5429.10,2.90,2766,197.60,60.20,2.70,2802,200.10,60.50
27,2026-11-01,Sonoma-Marin Area Rail Transit District,04 - Bay Area / Oakland,Bay Area 511 Sonoma-Marin Area Rail Transit VehiclePositions,Bay Area 511 Sonoma-Marin Area Rail Transit TripUpdates,624,Weekday,44.60,858.00,1,14,2,28,624.00,3.00,629,44.90,100.00,3.00,637,45.50,100.00
42,2026-11-01,Sonoma County Transit Schedule,04 - Bay Area / Oakland,Bay Area 511 Sonoma County Transit Vehicle Positions,Bay Area 511 Sonoma County Transit TripUpdates,2277,Weekday,227.70,2208.60,26,10,63,1008,10910.70,2.70,2243,224.30,98.20,2.00,2374,237.40,98.80
48,2026-11-01,Solano Transportation Authority,04 - Bay Area / Oakland,Bay Area 511 SolTrans VehiclePositions,Bay Area 511 SolTrans TripUpdates,4360,Weekday,311.40,3332.20,15,14,45,307,6065.90,3.00,4147,296.20,94.80,3.00,4371,312.20,98.70
33,2026-11-01,Santa Clara Valley Transportation Authority,04 - Bay Area / Oakland,Bay Area 511 Santa Clara Transit VehiclePositions,Bay Area 511 Santa Clara Transit TripUpdates,51355,Weekday,3668.20,52707.30,67,14,243,3228,158738.80,3.00,45163,3225.90,87.60,2.90,51203,3657.40,97.60
39,2026-11-01,San Mateo County Transit District,04 - Bay Area / Oakland,Bay Area 511 SamTrans VehiclePositions,Bay Area 511 SamTrans Trip Updates,23153,Weekday,1653.80,21902.60,75,14,336,1853,58139.10,3.00,22588,1613.40,97.20,3.00,22660,1618.60,97.50
30,2026-11-01,San Francisco Bay Ferry and Oakland Alameda Water Shuttle Schedule,04 - Bay Area / Oakland,Bay Area 511 San Francisco Bay Ferry VehiclePositions,Bay Area 511 San Francisco Bay Ferry TripUpdates,2518,Weekday,179.90,1130.20,6,14,26,11,368.10,3.00,2445,174.60,96.90,3.00,2532,180.90,99.40
9,2026-11-01,Peninsula Corridor Joint Powers Board,04 - Bay Area / Oakland,Bay Area 511 Caltrain VehiclePositions,Bay Area 511 Caltrain TripUpdates,1456,Weekday,104.00,1809.20,4,14,13,58,1989.00,2.50,1351,96.50,92.20,2.60,1361,97.20,92.70
57,2026-11-01,Napa Valley Transportation Authority,04 - Bay Area / Oakland,Bay Area 511 Vine Transit VehiclePositions,Bay Area 511 Vine Transit Trip Updates,3689,Weekday,263.50,3181.80,14,14,29,292,5889.50,2.90,2866,204.70,77.50,3.00,3597,256.90,96.80
21,2026-11-01,Marin County Transit District,04 - Bay Area / Oakland,Bay Area 511 Marin VehiclePositions,Bay Area 511 Marin TripUpdates,8115,Weekday,579.60,5880.70,20,14,54,543,14876.90,2.60,6246,446.10,76.70,2.60,8195,585.40,99.70


In [21]:
gtfs_table = utils.create_operator_table(
        operator_df, district_col = "Caltrans District"
    ).drop(columns = "Caltrans District")

In [22]:
gtfs_table

,Operator,Daily Trips,# Routes,# Shapes,# Stops,Daily Arrivals,Arrivals per Stop
0,Alameda-Contra Costa Transit District,4766,123,314,4679,217310,46.40
1,Central Contra Costa Transit Authority,866,42,89,1212,22538,18.60
2,City and County of San Francisco,8803,68,279,3239,325631,100.50
3,City of Fairfield,159,5,11,124,2484,20.00
4,City of Petaluma,221,11,20,152,3001,19.70
5,City of Santa Rosa,420,15,33,400,10406,26.00
6,City of Union City,251,6,16,167,6503,38.90
7,City of Vacaville,158,5,12,190,3544,18.70
8,Eastern Contra Costa Transit Authority,322,11,22,408,10775,26.40
9,"Golden Gate Bridge, Highway and Transportation District",256,10,23,246,6700,27.20


## `Legislative District Prep`

### sjoin_shapes_legislative_districts

In [23]:
from update_vars import DIGEST_DICT, PROCESSED_GCS, RAW_GCS, SHARED_GCS, abbrev_month

In [24]:
monthly_routes = gpd.read_parquet(
        f"{RAW_GCS}{DIGEST_DICT.route_map}_{abbrev_month}.parquet",
        storage_options={"token": credentials.token},
    columns=["name", "geometry"],
    )

In [25]:
monthly_routes.head(1).drop(columns = ["geometry"])

,name
0,Santa Maria Schedule


In [26]:
legislative_districts = gpd.read_parquet(
        f"{SHARED_GCS}legislative_districts.parquet",
        storage_options={"token": credentials.token},
    )

In [27]:
legislative_districts.sample().drop(columns = ["geometry"])

,legislative_district
61,AD 62


In [28]:
crosswalk = (
        gpd.sjoin(
            monthly_routes, legislative_districts, how="inner", predicate="intersects"
        )[["name", "legislative_district"]]
        .drop_duplicates()
        .sort_values(["name", "legislative_district"])
        .reset_index(drop=True)
    )

In [29]:
crosswalk.head(2)

,name,legislative_district
0,AC Transit Schedule,AD 14
1,AC Transit Schedule,AD 15


In [30]:
(
        f"{PROCESSED_GCS}{DIGEST_DICT.crosswalk_legislative}.parquet"
    )

'gs://calitp-analytics-data/data-analyses/gtfs_digest/processed/crosswalk_transit_operators_legislative_districts.parquet'

In [31]:
crosswalk_url = (
        f"{PROCESSED_GCS}{DIGEST_DICT.crosswalk_legislative}.parquet"
    )

legislative_districts_list = (
        pd.read_parquet(
            crosswalk_url,
            columns=["legislative_district"],
            filesystem=gcsfs.GCSFileSystem(),
        )
        .legislative_district.unique()
        .tolist()
    )


### AD 75 - no operators found in f"{PROCESSED_GCS}{DIGEST_DICT.operator_summary}.parquet"

'gs://calitp-analytics-data/data-analyses/gtfs_digest/processed/fct_monthly_operator_summary.parquet'

In [33]:
operator_raw_df = pd.read_parquet("gs://calitp-analytics-data/data-analyses/gtfs_digest/raw/fct_monthly_operator_summary_2026_10.parquet",
    filesystem = gcsfs.GCSFileSystem(),
    )

In [36]:
operator_processed_df = pd.read_parquet(f"{PROCESSED_GCS}{DIGEST_DICT.operator_summary}.parquet",
    filesystem = gcsfs.GCSFileSystem(),
    )

In [43]:
operator_raw_df.columns

Index(['month', 'year', 'month_first_day', 'schedule_name',
       'schedule_base64_url', 'vp_name', 'vp_base64_url', 'tu_name',
       'tu_base64_url', 'day_type', 'n_trips', 'n_trip_ids', 'daily_trips',
       'daily_trip_ids', 'ttl_service_hours', 'daily_service_hours',
       'n_routes', 'n_shapes', 'n_stops', 'ttl_arrivals', 'daily_arrivals',
       'n_flex_trips', 'daily_flex_trips', 'ttl_flex_service_hours',
       'daily_flex_service_hours', 'ttl_flex_arrivals', 'daily_flex_arrivals',
       'n_demand_response_trips', 'daily_demand_response_trips',
       'vp_messages_per_minute', 'n_vp_trips', 'daily_vp_trips',
       'pct_vp_trips', 'pct_vp_routes', 'daily_vp_extract_duration_minutes',
       'tu_messages_per_minute', 'n_tu_trips', 'daily_tu_trips',
       'pct_tu_trips', 'pct_tu_routes', 'daily_tu_extract_duration_minutes',
       'pct_tu_complete_minutes', 'pct_tu_accurate_minutes',
       'avg_prediction_spread_minutes', 'n_predictions',
       'pct_predictions_early', 'pc

In [35]:
operator_raw_df.loc[operator_raw_df.schedule_name.str.contains("North County")].shape

(9, 60)

In [45]:
operator_raw_df = operator_raw_df.fillna("0")

In [47]:
operator_raw_df.loc[operator_raw_df.tu_name.str.contains("North County")].shape

(0, 60)

In [ ]:
operator_raw_df.loc[operator_raw_df.vp_name.str.contains("North County")].shape

In [38]:
operator_processed_df.columns

Index(['Date', 'Analysis Name', 'Caltrans District', 'VP Name', 'TU Name',
       'N Trips', 'Day Type', 'Daily Trips', 'Ttl Service Hours', 'N Routes',
       'N Days', 'N Shapes', 'N Stops', 'Daily Arrivals',
       'VP Messages Per Minute', 'N VP Trips', 'Daily VP Trips',
       'Pct VP Trips', 'TU Messages Per Minute', 'N TU Trips',
       'Daily TU Trips', 'Pct TU Trips'],
      dtype='object')

In [39]:
operator_processed_df.loc[operator_processed_df["Analysis Name"].str.contains("North County")].shape

(0, 22)

In [40]:
operator_processed_df.loc[operator_processed_df["VP Name"].str.contains("North County")].shape

(0, 22)

In [41]:
operator_processed_df.loc[operator_processed_df["TU Name"].str.contains("North County")].shape

(0, 22)

In [51]:
crosswalk_url = f"{PROCESSED_GCS}{DIGEST_DICT.crosswalk}.parquet"

crosswalk_df = pd.read_parquet(
        crosswalk_url,
        columns=["name", "analysis_name"],
        filesystem=gcsfs.GCSFileSystem(),
    ).drop_duplicates()


In [52]:
crosswalk_df.head(2)

,name,analysis_name
0,Bay Area 511 Santa Clara Transit Schedule,Santa Clara Valley Transportation Authority
1,Cerritos on Wheels Schedule,City of Cerritos


In [54]:
crosswalk_df.loc[crosswalk_df.name.str.contains("North County")]

,name,analysis_name
192,North County Schedule,North County Transit District


In [59]:
import _prep_gtfs_data

In [56]:
filename = DIGEST_DICT.operator_summary

In [57]:
f"{RAW_GCS}{filename}_{abbrev_month}.parquet"

'gs://calitp-analytics-data/data-analyses/gtfs_digest/raw/fct_monthly_operator_summary_2026_10.parquet'

In [60]:
df = (
        pd.read_parquet(
            f"{RAW_GCS}{filename}_{abbrev_month}.parquet",
            storage_options={"token": credentials.token}
        )
        .rename(columns={"schedule_name": "name"})
        .pipe(
            _prep_gtfs_data.merge_with_crosswalk, columns=["name", "analysis_name", "caltrans_district"]
        )
    )

In [63]:
df.loc[df.analysis_name.str.contains("North County")].shape

(9, 62)

In [64]:
df2 = df[
        [
            "month_first_day",
            "analysis_name",
            "caltrans_district",
            "vp_name",
            "tu_name",
            "n_trips",
            "day_type",
            "daily_trips",
            "ttl_service_hours",  # "daily_service_hours",
            "n_routes",
            "n_days",
            "n_shapes",
            "n_stops",
            "daily_arrivals",
            "vp_messages_per_minute",
            "n_vp_trips",
            "daily_vp_trips",
            "pct_vp_trips",
            "tu_messages_per_minute",
            "n_tu_trips",
            "daily_tu_trips",
            "pct_tu_trips",
        ]
    ]

In [65]:
df2.loc[df2.analysis_name.str.contains("North County")].shape

(9, 22)

In [67]:
df2.loc[df2.analysis_name.str.contains("North County")]

,month_first_day,analysis_name,caltrans_district,vp_name,tu_name,n_trips,day_type,daily_trips,ttl_service_hours,n_routes,n_days,n_shapes,n_stops,daily_arrivals,vp_messages_per_minute,n_vp_trips,daily_vp_trips,pct_vp_trips,tu_messages_per_minute,n_tu_trips,daily_tu_trips,pct_tu_trips
84,2026-11-01,North County Transit District,11 - San Diego,None,None,0,Sunday,0.00,0.00,0,3,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN
120,2026-10-01,North County Transit District,11 - San Diego,None,None,0,Weekday,0.00,0.00,0,22,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN
183,2026-11-01,North County Transit District,11 - San Diego,None,None,0,Weekday,0.00,0.00,0,14,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN
194,2026-11-01,North County Transit District,11 - San Diego,None,None,0,Saturday,0.00,0.00,0,2,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN
204,2026-09-01,North County Transit District,11 - San Diego,None,None,0,Sunday,0.00,0.00,0,4,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN
249,2026-09-01,North County Transit District,11 - San Diego,None,None,0,Weekday,0.00,0.00,0,22,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN
420,2026-10-01,North County Transit District,11 - San Diego,None,None,0,Saturday,0.00,0.00,0,5,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN
460,2026-10-01,North County Transit District,11 - San Diego,None,None,0,Sunday,0.00,0.00,0,4,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN
490,2026-09-01,North County Transit District,11 - San Diego,None,None,0,Saturday,0.00,0.00,0,4,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN


In [68]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1887 entries, 0 to 1886
Data columns (total 22 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   month_first_day         1887 non-null   object 
 1   analysis_name           1887 non-null   object 
 2   caltrans_district       1887 non-null   object 
 3   vp_name                 731 non-null    object 
 4   tu_name                 777 non-null    object 
 5   n_trips                 1887 non-null   int64  
 6   day_type                1887 non-null   object 
 7   daily_trips             1887 non-null   float64
 8   ttl_service_hours       1887 non-null   float64
 9   n_routes                1887 non-null   int64  
 10  n_days                  1887 non-null   int64  
 11  n_shapes                1887 non-null   int64  
 12  n_stops                 1887 non-null   int64  
 13  daily_arrivals          1887 non-null   float64
 14  vp_messages_per_minute  730 non-null    

In [85]:
df3 = df2.sort_values(by = ["analysis_name","month_first_day","day_type", "n_trips"], ascending = [True, True, False, False]).drop_duplicates(subset = ["analysis_name","month_first_day","day_type",])

In [87]:
df3.loc[df3.analysis_name.str.contains("Alameda-Contra Costa")].sort_values(by = ["month_first_day"]).head(6)

,month_first_day,analysis_name,caltrans_district,vp_name,tu_name,n_trips,day_type,daily_trips,ttl_service_hours,n_routes,n_days,n_shapes,n_stops,daily_arrivals,vp_messages_per_minute,n_vp_trips,daily_vp_trips,pct_vp_trips,tu_messages_per_minute,n_tu_trips,daily_tu_trips,pct_tu_trips
1192,2026-09-01,Alameda-Contra Costa Transit District,04 - Bay Area / Oakland,Bay Area 511 AC Transit Vehicle Position,Bay Area 511 AC Transit Trip Updates,111943,Weekday,5088.30,91841.90,123,22,314,4679,232266.10,2.90,101176,4598.90,0.90,2.90,110354,5016.10,0.99
1188,2026-09-01,Alameda-Contra Costa Transit District,04 - Bay Area / Oakland,Bay Area 511 AC Transit Vehicle Position,Bay Area 511 AC Transit Trip Updates,16000,Sunday,4000.00,12490.70,60,4,152,3889,190409.00,3.00,13492,3373.00,0.84,3.00,15573,3893.30,0.97
1187,2026-09-01,Alameda-Contra Costa Transit District,04 - Bay Area / Oakland,Bay Area 511 AC Transit Vehicle Position,Bay Area 511 AC Transit Trip Updates,15992,Saturday,3998.00,12534.00,60,4,147,3889,190276.00,3.00,14055,3513.80,0.88,2.90,15594,3898.50,0.97
1193,2026-10-01,Alameda-Contra Costa Transit District,04 - Bay Area / Oakland,Bay Area 511 AC Transit Vehicle Position,Bay Area 511 AC Transit Trip Updates,113016,Weekday,5137.10,92909.20,123,22,314,4679,234178.60,2.90,99083,4503.80,0.88,2.90,110333,5015.10,0.98
1190,2026-10-01,Alameda-Contra Costa Transit District,04 - Bay Area / Oakland,Bay Area 511 AC Transit Vehicle Position,Bay Area 511 AC Transit Trip Updates,16000,Sunday,4000.00,12490.70,60,4,152,3889,190409.00,3.00,13231,3307.80,0.83,2.90,15360,3840.00,0.96
1186,2026-10-01,Alameda-Contra Costa Transit District,04 - Bay Area / Oakland,Bay Area 511 AC Transit Vehicle Position,Bay Area 511 AC Transit Trip Updates,19990,Saturday,3998.00,15667.50,60,5,147,3889,190276.00,3.00,16977,3395.40,0.85,2.90,19395,3879.00,0.97


In [88]:
df3.loc[df3.analysis_name.str.contains("North County")].sort_values(by = ["month_first_day"]).head(6)

,month_first_day,analysis_name,caltrans_district,vp_name,tu_name,n_trips,day_type,daily_trips,ttl_service_hours,n_routes,n_days,n_shapes,n_stops,daily_arrivals,vp_messages_per_minute,n_vp_trips,daily_vp_trips,pct_vp_trips,tu_messages_per_minute,n_tu_trips,daily_tu_trips,pct_tu_trips
249,2026-09-01,North County Transit District,11 - San Diego,None,None,0,Weekday,0.00,0.00,0,22,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN
204,2026-09-01,North County Transit District,11 - San Diego,None,None,0,Sunday,0.00,0.00,0,4,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN
490,2026-09-01,North County Transit District,11 - San Diego,None,None,0,Saturday,0.00,0.00,0,4,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN
120,2026-10-01,North County Transit District,11 - San Diego,None,None,0,Weekday,0.00,0.00,0,22,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN
460,2026-10-01,North County Transit District,11 - San Diego,None,None,0,Sunday,0.00,0.00,0,4,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN
420,2026-10-01,North County Transit District,11 - San Diego,None,None,0,Saturday,0.00,0.00,0,5,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN


In [94]:
crosswalk_url = (
    f"{PROCESSED_GCS}"
    f"{DIGEST_DICT.crosswalk_legislative}_2026_04.parquet"
)

filesystem = gcsfs.GCSFileSystem()

In [95]:
crosswalk_legislative = pd.read_parquet(
        crosswalk_url,
        filters=[
            ("legislative_district", "==", "AD 75"),
        ],
        filesystem=filesystem,
    )

In [96]:
operator_summary_url = f"{PROCESSED_GCS}{DIGEST_DICT.operator_summary}.parquet"

operator_df = pd.read_parquet(
    operator_summary_url,
    filesystem = gcsfs.GCSFileSystem(),
    filters=[
        ("Day Type", "==", "Weekday"),
        ("Analysis Name", "in", ["North County Transit District"]),
    ],
    ).merge(
    crosswalk_legislative.rename(columns = {"analysis_name": "Analysis Name"}),
    on = "Analysis Name",
    how = "inner"
    ).reset_index(drop=True)

In [97]:
operator_df

,Date,Analysis Name,Caltrans District,VP Name,TU Name,N Trips,Day Type,Daily Trips,Ttl Service Hours,N Routes,N Days,N Shapes,N Stops,Daily Arrivals,VP Messages Per Minute,N VP Trips,Daily VP Trips,Pct VP Trips,TU Messages Per Minute,N TU Trips,Daily TU Trips,Pct TU Trips,legislative_district
0,2026-09-01,North County Transit District,11 - San Diego,None,None,0,Weekday,0.00,0.00,0,22,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN,AD 75
1,2026-10-01,North County Transit District,11 - San Diego,None,None,0,Weekday,0.00,0.00,0,22,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN,AD 75
2,2026-11-01,North County Transit District,11 - San Diego,None,None,0,Weekday,0.00,0.00,0,14,0,0,0.00,NaN,0,0.00,NaN,NaN,0,0.00,NaN,AD 75


In [98]:
fct_monthly_routes_url = f"{PROCESSED_GCS}{DIGEST_DICT.route_map}.parquet"

fct_monthly_route_df = gpd.read_parquet(
        fct_monthly_routes_url,
        storage_options = {"token": credentials.token},
    ).reset_index(drop=True).pipe(
        utils.prep_gdf
    )

In [100]:
fct_monthly_route_df.sample().drop(columns = ["Geometry"])

,Analysis Name,Route Name,Route Length Miles
847,Eastern Contra Costa Transit Authority,372__372 Pittsburg-Bay Point BART/ Los Medanos College Pittsburg,11.80


In [101]:
fct_monthly_route_df.loc[fct_monthly_route_df["Analysis Name"].str.contains("North County")].shape

(0, 4)